# Aprendizado supervisionado clássico e quântico: Iris

O objetivo é comparar, em condições controladas e idênticas, classificadores clássicos e quânticos simulados em uma tarefa binária pequena. A tarefa escolhida foi distinguir *Iris versicolor* de *Iris virginica* pelos quatro atributos da base de dados. Os modelos comparados são:

- uma referência trivial (`DummyClassifier`);
- SVM linear;
- SVM com kernel RBF;
- um MLP pequeno;
- uma SVM com kernel quântico de fidelidade (QSVM);
- um classificador variacional híbrido (VQC).

Também há um experimento complementar de treinabilidade (platôs estéreis).

**Não há hipótese de vantagem quântica.** Os resultados descrevem o comportamento dos modelos neste recorte de dados, com simulação exata de 4 qubits em PennyLane (`default.qubit`). Um resultado inferior ou semelhante ao clássico é um resultado válido. Nenhuma semente, classe, transformação ou subconjunto foi escolhida após observar o conjunto de teste.

O notebook é autocontido. Todas as funções são definidas aqui, e ele não depende de outros notebooks nem de resultados gravados por eles.

## Sumário

0. [Configuração](#sec-0)
1. [Formulação do aprendizado supervisionado](#sec-1)
2. [Dados: aquisição, integridade e exploração](#sec-2)
3. [Protocolo experimental: divisões, representação e barreiras contra vazamento](#sec-3)
4. [Modelos clássicos](#sec-4)
5. [QSVM com kernel de fidelidade](#sec-5)
6. [Classificador variacional (VQC) e relação com o MLP](#sec-6)
7. [Experimento de treinabilidade: platôs estéreis](#sec-7)
8. [Comparação final](#sec-8)
9. [Verificações, ambiente e tempo de execução](#sec-9)
10. [Referências](#sec-10)

<a id="sec-0"></a>
## 0. Configuração

O modo `rapido` é o padrão. O modo `completo` usa várias sementes de divisão e mais épocas e amostras. Para trocar de modo, edite `MODO` ou defina a variável de ambiente `TCC_MODO=completo` antes de iniciar o kernel.

In [ ]:
from __future__ import annotations

import copy
import hashlib
import io
import itertools
import math
import os
import platform
import time
import urllib.request
import warnings
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Callable, Sequence

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import scipy
import sklearn
from pennylane import numpy as pnp
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, log_loss)
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

T_INICIO_NOTEBOOK = time.perf_counter()
MODO = os.environ.get("TCC_MODO", "rapido")  # "rapido" ou "completo"

if MODO not in {"rapido", "completo"}:
    raise ValueError("MODO deve ser 'rapido' ou 'completo'.")

SEMENTE = 20261119
CONFIG_RAPIDO = {
    "sementes_divisao": [SEMENTE],
    "fracao_validacao": 0.2,
    "fracao_teste": 0.2,
    "subamostra": None,
    "svm_linear_C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "svm_rbf_C": [0.1, 1.0, 10.0, 100.0],
    "svm_rbf_gamma": ["scale", 0.01, 0.1, 1.0],
    "mlp_ocultas": [(4,), (8,), (16,)],
    "mlp_alpha": [1e-4, 1e-2],
    "mlp_epocas": 300,
    "qsvm_camadas": [1, 2],
    "qsvm_C": [0.1, 1.0, 10.0, 100.0],
    "qsvm_lote": 4096,
    "shots_lista": [100, 1000, 10000],
    "shots_pares": 150,
    "vqc_camadas": [1, 2, 3],
    "vqc_epocas": 100,
    "vqc_taxa": 0.05,
    "bp_qubits": [2, 4, 6, 8, 10],
    "bp_amostras": 1000,
    "bp_ext_qubits": [2, 4, 6, 8],
    "bp_ext_profundidades": [1, 2, 4, 8],
    "bp_ext_amostras": 150,
}
CONFIG_COMPLETO = {
    **CONFIG_RAPIDO,
    "sementes_divisao": [SEMENTE + k for k in range(5)],
    "subamostra": None,
    "mlp_epocas": 500,
    "vqc_epocas": 200,
    "bp_amostras": 5000,
    "bp_ext_amostras": 500,
}
CONFIG = CONFIG_RAPIDO if MODO == "rapido" else CONFIG_COMPLETO
N_QUBITS = 4

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_FIG = RAIZ / "outputs" / "02_iris" / "figuras"
DIR_TAB = RAIZ / "outputs" / "02_iris" / "tabelas"
DIR_FIG.mkdir(parents=True, exist_ok=True)
DIR_TAB.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 10, "axes.titlesize": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "legend.fontsize": 9})
DEV = qml.device("default.qubit")
TOL = 1e-10
print(f"Modo: {MODO} | Python {platform.python_version()} | PennyLane {qml.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
def salvar_figura(fig: plt.Figure, nome: str) -> None:
    fig.savefig(DIR_FIG / f"{nome}.png", bbox_inches="tight")


def salvar_tabela(tabela: pd.DataFrame, nome: str) -> None:
    tabela.to_csv(DIR_TAB / f"{nome}.csv", index=False)


REGISTRO_VERIFICACOES: list[dict] = []


def verificar(nome: str, condicao: bool, detalhe: str = "") -> None:
    """Registra uma verificação e interrompe a execução se ela falhar."""
    REGISTRO_VERIFICACOES.append({"verificação": nome, "aprovada": bool(condicao), "detalhe": detalhe})
    if not condicao:
        raise AssertionError(f"Verificação falhou: {nome}. {detalhe}")

<a id="sec-1"></a>
## 1. Formulação do aprendizado supervisionado

Temos exemplos $(\mathbf x_i, y_i)$, com atributos $\mathbf x_i \in \mathbb R^d$ e rótulos $y_i \in \{0, 1\}$. Queremos ajustar uma função $f(\mathbf x)$ que preveja o rótulo de exemplos novos. Os dados são divididos em três conjuntos disjuntos:

| Conjunto | Uso |
| --- | --- |
| Treinamento | Ajustar parâmetros: pesos, vetores de suporte, normalizações, PCA |
| Validação | Escolher hiperparâmetros e o ponto de parada; nunca usado para ajustar parâmetros |
| Teste | Avaliação final, uma única vez por modelo selecionado |

A divisão é feita antes de qualquer ajuste de normalização, PCA ou hiperparâmetro. Todas as transformações são ajustadas apenas com o conjunto de treinamento. O teste só é usado na Seção 8, depois que todos os modelos foram selecionados.

**Métricas.** Seja $TP$, $TN$, $FP$, $FN$ a contagem de verdadeiros e falsos positivos e negativos, com a classe 1 como positiva.

- Acurácia: $(TP+TN)/n$.
- Acurácia balanceada: média das revocações por classe, $\tfrac12\big(\tfrac{TP}{TP+FN} + \tfrac{TN}{TN+FP}\big)$. Vale $0{,}5$ para um classificador constante.
- F1: média harmônica entre precisão e revocação da classe 1.

A acurácia balanceada é o critério de seleção na validação.

<a id="sec-2"></a>
## 2. Dados: Iris

### 2.1 Origem e licença

O conjunto Iris contém 150 flores de três espécies (*setosa*, *versicolor*, *virginica*), 50 de cada, com quatro atributos contínuos em centímetros: comprimento e largura da sépala e da pétala.

Usamos o carregador `sklearn.datasets.load_iris`, que traz os dados embutidos e dispensa download.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df_iris = iris.frame.rename(columns={
    "sepal length (cm)": "comprimento da sépala (cm)", "sepal width (cm)": "largura da sépala (cm)",
    "petal length (cm)": "comprimento da pétala (cm)", "petal width (cm)": "largura da pétala (cm)"})
df_iris["espécie"] = df_iris["target"].map(dict(enumerate(iris.target_names)))
NOMES_ATRIBUTOS = list(df_iris.columns[:4])

verificar("Iris: 150 linhas, 4 atributos numéricos, sem ausentes",
          len(df_iris) == 150 and df_iris[NOMES_ATRIBUTOS].dtypes.map(pd.api.types.is_float_dtype).all()
          and not df_iris.isna().any().any())
print(df_iris["espécie"].value_counts().to_string())
df_iris[NOMES_ATRIBUTOS].describe().round(2)

### 2.2 Exploração das três espécies

A matriz de dispersão mostra que *setosa* se separa das demais pelos atributos da pétala, enquanto *versicolor* e *virginica* se sobrepõem parcialmente. Para manter uma comparação binária simples e não trivial, a tarefa é distinguir versicolor (rótulo 0) de virginica (rótulo 1) usando os quatro atributos. Essa escolha foi fixada antes de qualquer modelagem.

In [ ]:
fig, eixos = plt.subplots(4, 4, figsize=(11, 10))
cores = {"setosa": "C2", "versicolor": "C0", "virginica": "C3"}

for i, a in enumerate(NOMES_ATRIBUTOS):

    for j, b in enumerate(NOMES_ATRIBUTOS):
        eixo = eixos[i, j]

        for especie, grupo in df_iris.groupby("espécie"):

            if i == j:
                eixo.hist(grupo[a], bins=12, alpha=0.6, color=cores[especie], label=especie)
            else:
                eixo.scatter(grupo[b], grupo[a], s=8, alpha=0.7, color=cores[especie], label=especie)

        if i == 3:
            eixo.set_xlabel(b, fontsize=8)
        if j == 0:
            eixo.set_ylabel(a, fontsize=8)
        eixo.tick_params(labelsize=7)

eixos[0, 0].legend(fontsize=7)
fig.suptitle("Iris: atributos por espécie (diagonal: histogramas; fora da diagonal: dispersão)")
fig.tight_layout()
salvar_figura(fig, "exploracao_atributos")
plt.show()

### 2.3 Integridade, duplicatas e rótulos

Registros idênticos podem cair em conjuntos diferentes e inflar a avaliação. A política adotada é aplicada antes da divisão:

- duplicatas exatas são reduzidas a uma ocorrência;
- vetores de atributos idênticos com rótulos diferentes são removidos, por serem ambíguos.

In [ ]:
df_tarefa = df_iris[df_iris["espécie"].isin(["versicolor", "virginica"])].copy()
df_tarefa["rótulo"] = (df_tarefa["espécie"] == "virginica").astype(int)
ROTULOS_CLASSES = {0: "versicolor", 1: "virginica"}
DESCRICAO_TAREFA = "versicolor (0) contra virginica (1)"


def remover_duplicatas(df: pd.DataFrame, atributos: list[str], rotulo: str) -> tuple[pd.DataFrame, dict]:
    """Remove duplicatas exatas e vetores de atributos idênticos com rótulos conflitantes."""
    n0 = len(df)
    sem_exatas = df.drop_duplicates(subset=atributos + [rotulo])
    rotulos_por_vetor = sem_exatas.groupby(atributos)[rotulo].transform("nunique")
    limpo = sem_exatas[rotulos_por_vetor == 1].reset_index(drop=True)

    return limpo, {"linhas originais": n0, "duplicatas exatas removidas": n0 - len(sem_exatas),
                   "linhas com rótulos conflitantes removidas": len(sem_exatas) - len(limpo),
                   "linhas finais": len(limpo)}


df_limpo, relatorio_duplicatas = remover_duplicatas(df_tarefa, NOMES_ATRIBUTOS, "rótulo")
verificar("sem vetores de atributos repetidos após a limpeza", not df_limpo.duplicated(subset=NOMES_ATRIBUTOS).any())
X_bruto = df_limpo[NOMES_ATRIBUTOS].to_numpy(dtype=float)
y_bruto = df_limpo["rótulo"].to_numpy(dtype=int)
print(relatorio_duplicatas)
print("Distribuição das classes na tarefa:", {ROTULOS_CLASSES[k]: int(v) for k, v in zip(*np.unique(y_bruto, return_counts=True))})

### 2.4 Representação de entrada dos modelos

Com quatro atributos, cada um é codificado em um qubit. A transformação é um `MinMaxScaler` ajustado somente no treinamento, que leva cada atributo ao intervalo $[0, \pi]$. Valores novos fora do intervalo observado no treinamento são recortados para $[0, \pi]$ (`clip=True`). No pior caso, um valor extremo é tratado como o extremo do treinamento.

O intervalo $[0, \pi]$ evita a periodicidade das rotações: $R_Y(\theta)|0\rangle = \cos\tfrac\theta2|0\rangle + \sin\tfrac\theta2|1\rangle$ é injetiva em $\theta \in [0,\pi]$. Já o intervalo $[0, 2\pi]$ levaria os dois extremos ao mesmo estado, a menos de fase global. Todos os modelos, clássicos e quânticos, recebem exatamente os mesmos valores em $[0,\pi]^4$.

In [ ]:
DESCRICAO_REPRESENTACAO = "4 atributos originais -> MinMaxScaler([0, π], clip=True) ajustado no treino"

def ajustar_representacao(X_treino: np.ndarray) -> Pipeline:
    return Pipeline([("angulos", MinMaxScaler(feature_range=(0, np.pi), clip=True))]).fit(X_treino)

ajustar_controle = None  # Iris já usa os atributos originais

def relatorio_representacao(p: "Particao") -> None:
    escala = p.transformacao.named_steps["angulos"]
    print("Intervalos aprendidos no treino (cm):", {nome: (round(float(a), 2), round(float(b), 2)) for nome, a, b in zip(NOMES_ATRIBUTOS, escala.data_min_, escala.data_max_)})

<a id="sec-3"></a>
## 3. Protocolo experimental

### 3.1 Recorte, divisões e sementes

1. **Recorte.** Usamos todos os exemplos de versicolor e virginica que restam após a remoção de duplicatas (Seção 2.3), sem subamostragem.
2. **Divisão estratificada** em treinamento (60 %), validação (20 %) e teste (20 %), com a semente de divisão explícita. No modo completo, o protocolo inteiro é repetido para cada semente da lista `CONFIG["sementes_divisao"]`.
3. **Ajustes.** Transformações e modelos são ajustados no treinamento. Hiperparâmetros e ponto de parada são escolhidos pela acurácia balanceada na validação. Em empates, vence a primeira configuração da grade, na ordem declarada. O modelo selecionado, treinado apenas no treinamento, é avaliado uma vez no teste. Não há reajuste em treinamento mais validação, de modo que o procedimento é o mesmo para todos os modelos, inclusive o VQC.

In [ ]:
def subamostrar_estratificado(y: np.ndarray, tamanho: int | None, semente: int) -> np.ndarray:
    """Índices de uma subamostra estratificada fixa; todos os índices se tamanho for None."""
    indices = np.arange(len(y))
    if tamanho is None or tamanho >= len(y):
        return indices
    selecionados, _ = train_test_split(indices, train_size=tamanho, stratify=y, random_state=semente)
    return np.sort(selecionados)


def dividir_estratificado(y: np.ndarray, semente: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    indices = np.arange(len(y))
    f_te, f_val = CONFIG["fracao_teste"], CONFIG["fracao_validacao"]
    i_dev, i_te = train_test_split(indices, test_size=f_te, stratify=y, random_state=semente)
    i_tr, i_val = train_test_split(i_dev, test_size=f_val / (1 - f_te), stratify=y[i_dev], random_state=semente)
    return np.sort(i_tr), np.sort(i_val), np.sort(i_te)


@dataclass
class Particao:
    semente: int
    indices: dict[str, np.ndarray]
    Z: dict[str, np.ndarray]
    y: dict[str, np.ndarray]
    transformacao: Pipeline
    C: dict[str, np.ndarray] | None = None


def preparar_particao(X: np.ndarray, y: np.ndarray, semente: int) -> Particao:
    i_tr, i_val, i_te = dividir_estratificado(y, semente)
    indices = {"treino": i_tr, "validação": i_val, "teste": i_te}
    transformacao = ajustar_representacao(X[i_tr])  # ajuste apenas no treino
    Z = {k: transformacao.transform(X[i]) for k, i in indices.items()}
    C = None

    if ajustar_controle is not None:
        controle = ajustar_controle(X[i_tr])
        C = {k: controle.transform(X[i]) for k, i in indices.items()}

    return Particao(semente, indices, Z, {k: y[i] for k, i in indices.items()}, transformacao, C)


def verificar_particao(p: Particao, X: np.ndarray) -> None:
    i_tr, i_val, i_te = (p.indices[k] for k in ("treino", "validação", "teste"))
    verificar(f"semente {p.semente}: divisões disjuntas e completas", not (set(i_tr) & set(i_val) or set(i_tr) & set(i_te) or set(i_val) & set(i_te)) and len(i_tr) + len(i_val) + len(i_te) == len(X))

    hashes = {k: {hashlib.sha1(X[i].tobytes()).hexdigest() for i in idx} for k, idx in p.indices.items()}
    verificar(f"semente {p.semente}: nenhum vetor de atributos idêntico entre divisões", not (hashes["treino"] & hashes["teste"] or hashes["treino"] & hashes["validação"] or hashes["validação"] & hashes["teste"]))

    vistos = [getattr(etapa, "n_samples_seen_", getattr(etapa, "n_samples_", None)) for _, etapa in p.transformacao.steps]

    vistos = [int(np.max(v)) for v in vistos if v is not None]
    verificar(f"semente {p.semente}: transformações ajustadas apenas com o treino ({len(i_tr)} exemplos)", len(vistos) > 0 and all(v == len(i_tr) for v in vistos), f"amostras vistas: {vistos}")
    refeita = ajustar_representacao(X[i_tr])
    verificar(f"semente {p.semente}: representação do teste reproduzível a partir do treino", np.allclose(refeita.transform(X[i_te]), p.Z["teste"]))
    verificar(f"semente {p.semente}: representação dentro de [0, pi]", all(z.min() >= -1e-12 and z.max() <= np.pi + 1e-12 for z in p.Z.values()))


idx_recorte = subamostrar_estratificado(y_bruto, CONFIG["subamostra"], SEMENTE)
X_base, y_base = X_bruto[idx_recorte], y_bruto[idx_recorte]
particoes = {s: preparar_particao(X_base, y_base, s) for s in CONFIG["sementes_divisao"]}

for p in particoes.values():
    verificar_particao(p, X_base)

P0 = particoes[CONFIG["sementes_divisao"][0]]

linhas = []

for nome, yy in P0.y.items():
    linhas.append({"conjunto": nome, "n": len(yy), f"classe 0 ({ROTULOS_CLASSES[0]})": int((yy == 0).sum()),
                   f"classe 1 ({ROTULOS_CLASSES[1]})": int((yy == 1).sum()), "fração da classe 1": (yy == 1).mean()})

tabela_divisoes = pd.DataFrame(linhas)
salvar_tabela(tabela_divisoes, "divisoes")
print(f"Recorte usado: {len(y_base)} exemplos. Representação: {DESCRICAO_REPRESENTACAO}")
relatorio_representacao(P0)
tabela_divisoes.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.8))
largura = 0.38
pos = np.arange(len(tabela_divisoes))

for k, cor in ((0, "C0"), (1, "C3")):
    col = [c for c in tabela_divisoes.columns if c.startswith(f"classe {k}")][0]
    eixos[0].bar(pos + (k - 0.5) * largura, tabela_divisoes[col], width=largura, color=cor, label=col)

eixos[0].set(xticks=pos, xticklabels=tabela_divisoes["conjunto"], ylabel="número de exemplos", title=f"Distribuição das classes nas divisões (semente {P0.semente})")
eixos[0].legend()

for k, cor in ((0, "C0"), (1, "C3")):
    eixos[1].hist(P0.Z["treino"][P0.y["treino"] == k].ravel(), bins=20, alpha=0.6, color=cor, label=f"treino, classe {k}")

eixos[1].set(xlabel="valor apresentado aos modelos (ângulo, rad)", ylabel="contagem (todas as coordenadas)", title="Valores de entrada após a transformação ajustada no treino", xlim=(0, np.pi))
eixos[1].legend()
fig.tight_layout()
salvar_figura(fig, "divisoes_e_representacao")
plt.show()

### 3.2 Registro dos resultados

Cada modelo devolve a mesma estrutura de resultado: hiperparâmetros selecionados, métricas de validação e teste, número de parâmetros, tempo de parede medido no simulador e recursos quânticos contados. As métricas de teste são calculadas dentro das funções de ajuste, mas não participam de nenhuma decisão. A seleção usa apenas `metricas_validacao`.

In [ ]:
@dataclass
class ResultadoModelo:
    modelo: str
    hiperparametros: dict
    metricas_validacao: dict
    metricas_teste: dict
    y_pred_teste: np.ndarray
    parametros: int
    tempo_s: float
    recursos: dict = field(default_factory=dict)
    extras: dict = field(default_factory=dict)


def metricas(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {"acurácia": accuracy_score(y_true, y_pred), "acurácia balanceada": balanced_accuracy_score(y_true, y_pred), "F1": f1_score(y_true, y_pred, zero_division=0)}

<a id="sec-4"></a>
## 4. Modelos clássicos

### 4.1 Referência trivial

O `DummyClassifier(strategy="most_frequent")` sempre prevê a classe majoritária do treinamento. Sua acurácia balanceada é $0{,}5$ por construção. Ele estabelece o patamar mínimo que um modelo útil deve superar.

### 4.2 SVM linear

A SVM procura o hiperplano $\mathbf w^\top \mathbf z + b = 0$ de margem máxima $2/\lVert\mathbf w\rVert$ entre as classes, com folgas penalizadas por $C$:

$$\min_{\mathbf w, b, \xi}\ \tfrac12\lVert \mathbf w\rVert^2 + C\sum_i \xi_i \quad\text{s.a.}\quad y_i'(\mathbf w^\top\mathbf z_i + b) \ge 1 - \xi_i,\ \xi_i \ge 0,$$

com $y_i' = 2y_i - 1 \in \{-1, +1\}$. Na solução, $\mathbf w = \sum_i \alpha_i y_i' \mathbf z_i$, e apenas os exemplos com $\alpha_i > 0$, os vetores de suporte, determinam a fronteira. Contamos como parâmetros os coeficientes duais não nulos mais o viés.

### 4.3 SVM com kernel RBF e o truque do kernel

O problema dual depende dos dados apenas por produtos internos $\mathbf z_i^\top \mathbf z_j$. Substituí-los por um kernel $k(\mathbf z_i, \mathbf z_j) = \langle \phi(\mathbf z_i), \phi(\mathbf z_j)\rangle$ equivale a aplicar a SVM linear em um espaço de características $\phi(\mathbf z)$ sem calcular $\phi$ explicitamente. O kernel RBF $k(\mathbf z, \mathbf z') = e^{-\gamma\lVert \mathbf z - \mathbf z'\rVert^2}$ corresponde a um espaço de dimensão infinita. A QSVM da Seção 5 usa a mesma ideia, com $\phi$ dado por um estado quântico.

A ilustração abaixo é separada do dataset e usa dados sintéticos unidimensionais. Os pontos $x \in [-2, 2]$ têm classe $[|x| > 1]$, que não é linearmente separável na reta. O mapa $\phi(x) = (x, x^2)$ os torna linearmente separáveis.

In [ ]:
rng_ilustracao = np.random.default_rng(SEMENTE)
x_il = rng_ilustracao.uniform(-2, 2, 60)
y_il = (np.abs(x_il) > 1).astype(int)
fig, eixos = plt.subplots(1, 2, figsize=(11, 3.4))

for k, cor in ((0, "C0"), (1, "C3")):
    eixos[0].scatter(x_il[y_il == k], np.zeros((y_il == k).sum()), color=cor, label=f"classe {k}")
    eixos[1].scatter(x_il[y_il == k], x_il[y_il == k] ** 2, color=cor, label=f"classe {k}")

eixos[1].axhline(1, color="black", ls="--", lw=1, label="hiperplano x² = 1 no espaço φ")
eixos[0].set(xlabel="x", yticks=[], title="Espaço original (1D): não separável por um limiar")
eixos[1].set(xlabel="x", ylabel="x²", title="Espaço de características φ(x) = (x, x²)")

for e in eixos:
    e.legend(fontsize=8)

fig.suptitle("Ilustração do truque do kernel (dados sintéticos, separados dos resultados do dataset)")
fig.tight_layout()
salvar_figura(fig, "ilustracao_mapa_caracteristicas")
plt.show()

### 4.4 MLP pequeno

Um perceptron multicamadas com uma camada oculta calcula

$$\mathbf h = \operatorname{ReLU}(W_1\mathbf z + \mathbf b_1), \qquad p(y=1\mid\mathbf z) = \sigma(\mathbf w_2^\top\mathbf h + b_2),$$

e é treinado minimizando a entropia cruzada binária com regularização $L_2$ de peso $\alpha$:

$$\mathcal L = -\tfrac1n\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big] + \tfrac{\alpha}{2n}\lVert W\rVert^2.$$

A retropropagação calcula $\partial\mathcal L/\partial W$ pela regra da cadeia, da saída para a entrada, em uma única passagem reversa. O otimizador Adam atualiza os pesos. Treinamos época a época com `partial_fit` para registrar as curvas de treinamento e validação. Para cada configuração, guardamos os pesos da época de menor perda de validação, que funciona como parada antecipada.

### 4.5 Seleção de hiperparâmetros

Usamos grades pequenas e explícitas para todos os modelos, inclusive os clássicos, para não prejudicá-los por falta de ajuste. As grades estão em `CONFIG`.

In [ ]:
def selecionar_sklearn(nome: str, construtor: Callable[..., object], grade: list[dict], p: Particao, chave: str = "Z") -> ResultadoModelo:
    """Ajusta cada configuração no treino e escolhe pela acurácia balanceada na validação."""
    dados = getattr(p, chave)
    t0 = time.perf_counter()
    melhor, melhor_score, linhas = None, -np.inf, []

    for hp in grade:
        modelo = construtor(**hp).fit(dados["treino"], p.y["treino"])
        score = balanced_accuracy_score(p.y["validação"], modelo.predict(dados["validação"]))
        linhas.append({**hp, "acurácia balanceada (validação)": score})
        if score > melhor_score:
            melhor, melhor_score, melhor_hp = modelo, score, hp

    y_val = melhor.predict(dados["validação"])
    y_te = melhor.predict(dados["teste"])
    tempo = time.perf_counter() - t0

    if isinstance(melhor, SVC):
        n_par = int(melhor.dual_coef_.size + melhor.intercept_.size)
    elif isinstance(melhor, DummyClassifier):
        n_par = 0
    else:
        n_par = int(sum(w.size for w in melhor.coefs_) + sum(b.size for b in melhor.intercepts_))

    return ResultadoModelo(nome, melhor_hp, metricas(p.y["validação"], y_val), metricas(p.y["teste"], y_te), y_te, n_par, tempo, extras={"grade": pd.DataFrame(linhas), "modelo": melhor})


@dataclass
class MLPComCurvas:
    """MLP treinado por épocas; guarda os pesos da época de menor perda de validação."""
    ocultas: tuple
    alpha: float
    epocas: int
    semente: int
    curvas: dict = field(default_factory=dict)
    modelo: MLPClassifier | None = None

    def fit(self, Z_tr: np.ndarray, y_tr: np.ndarray, Z_val: np.ndarray, y_val: np.ndarray) -> "MLPComCurvas":
        base = MLPClassifier(hidden_layer_sizes=self.ocultas, alpha=self.alpha, learning_rate_init=0.01,
                             random_state=self.semente)
        perda_tr, perda_val, melhor_perda = [], [], np.inf

        for _ in range(self.epocas):
            base.partial_fit(Z_tr, y_tr, classes=np.array([0, 1]))
            perda_tr.append(log_loss(y_tr, base.predict_proba(Z_tr), labels=[0, 1]))
            perda_val.append(log_loss(y_val, base.predict_proba(Z_val), labels=[0, 1]))
            if perda_val[-1] < melhor_perda:
                melhor_perda, self.modelo = perda_val[-1], copy.deepcopy(base)

        self.curvas = {"treino": perda_tr, "validação": perda_val, "melhor época": int(np.argmin(perda_val)) + 1}

        return self

    def predict(self, Z: np.ndarray) -> np.ndarray:
        return self.modelo.predict(Z)


def ajustar_mlp(p: Particao, chave: str = "Z", nome: str = "MLP") -> ResultadoModelo:
    dados = getattr(p, chave)
    t0 = time.perf_counter()
    melhor, melhor_score, linhas = None, -np.inf, []

    for ocultas, alpha in itertools.product(CONFIG["mlp_ocultas"], CONFIG["mlp_alpha"]):
        m = MLPComCurvas(ocultas, alpha, CONFIG["mlp_epocas"], p.semente).fit(
            dados["treino"], p.y["treino"], dados["validação"], p.y["validação"])
        score = balanced_accuracy_score(p.y["validação"], m.predict(dados["validação"]))
        linhas.append({"camada oculta": ocultas, "alpha": alpha, "melhor época": m.curvas["melhor época"],
                       "acurácia balanceada (validação)": score})
        if score > melhor_score:
            melhor, melhor_score = m, score

    y_te = melhor.predict(dados["teste"])
    n_par = int(sum(w.size for w in melhor.modelo.coefs_) + sum(b.size for b in melhor.modelo.intercepts_))

    return ResultadoModelo(nome, {"camada oculta": melhor.ocultas, "alpha": melhor.alpha, "época selecionada": melhor.curvas["melhor época"]}, metricas(p.y["validação"], melhor.predict(dados["validação"])), metricas(p.y["teste"], y_te), y_te, n_par, time.perf_counter() - t0, extras={"grade": pd.DataFrame(linhas), "curvas": melhor.curvas})

def grade(**listas) -> list[dict]:
    chaves = list(listas)
    return [dict(zip(chaves, valores)) for valores in itertools.product(*listas.values())]

def ajustar_classicos(p: Particao, chave: str = "Z", sufixo: str = "") -> dict[str, ResultadoModelo]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)

        return {
            "Dummy" + sufixo: selecionar_sklearn("Dummy" + sufixo, DummyClassifier, [{"strategy": "most_frequent"}], p, chave),
            "SVM linear" + sufixo: selecionar_sklearn("SVM linear" + sufixo, SVC, grade(kernel=["linear"], C=CONFIG["svm_linear_C"]), p, chave),
            "SVM RBF" + sufixo: selecionar_sklearn("SVM RBF" + sufixo, SVC, grade(kernel=["rbf"], C=CONFIG["svm_rbf_C"], gamma=CONFIG["svm_rbf_gamma"]), p, chave),
            "MLP" + sufixo: ajustar_mlp(p, chave, "MLP" + sufixo),
        }


resultados_classicos = ajustar_classicos(P0)
pd.DataFrame([{"modelo": r.modelo, "hiperparâmetros selecionados": r.hiperparametros, **{f"{k} (validação)": v for k, v in r.metricas_validacao.items()}, "parâmetros": r.parametros, "tempo (s)": r.tempo_s} for r in resultados_classicos.values()]).round(3)

In [ ]:
curvas_mlp = resultados_classicos["MLP"].extras["curvas"]
fig, eixo = plt.subplots(figsize=(7, 3.8))
eixo.plot(np.arange(1, len(curvas_mlp["treino"]) + 1), curvas_mlp["treino"], label="treino")
eixo.plot(np.arange(1, len(curvas_mlp["validação"]) + 1), curvas_mlp["validação"], label="validação")
eixo.axvline(curvas_mlp["melhor época"], color="gray", ls="--", label=f"época selecionada ({curvas_mlp['melhor época']})")
eixo.set(xlabel="época", ylabel="entropia cruzada binária", title=f"MLP selecionado {resultados_classicos['MLP'].hiperparametros['camada oculta']}: curvas de perda")
eixo.legend()
fig.tight_layout()
salvar_figura(fig, "mlp_curvas")
plt.show()

<a id="sec-5"></a>
## 5. QSVM com kernel quântico de fidelidade

### 5.1 Mapa de características quântico

Cada entrada $\mathbf z \in [0,\pi]^4$ é codificada em um estado de 4 qubits, $|\phi(\mathbf z)\rangle = U(\mathbf z)|0000\rangle$, em que $U$ repete $\ell$ vezes a camada

$$\prod_{i} R_Y(z_i)\ \cdot\ \prod_{(i, i+1)} \mathrm{IsingZZ}\!\big(z_i z_{i+1}/\pi\big),$$

com os pares $(i, i+1)$ em anel. As rotações $R_Y$ codificam os atributos individualmente. As interações $ZZ$, com $\mathrm{IsingZZ}(\varphi) = e^{-i\varphi Z\otimes Z/2}$, introduzem termos de produto e emaranhamento. A divisão por $\pi$ mantém o ângulo das interações em $[0,\pi]$, para que extremos distintos não sejam identificados pela periodicidade.

O kernel de fidelidade é:

$$k(\mathbf z, \mathbf z') = |\langle \phi(\mathbf z')|\phi(\mathbf z)\rangle|^2.$$

### 5.2 Circuito Inversion Test (compute–uncompute)

O circuito prepara $U(\mathbf z)|0\rangle$, aplica $U(\mathbf z')^\dagger$ e mede todos os qubits. A probabilidade de obter $|0000\rangle$ é

$$P(0000) = |\langle 0|U(\mathbf z')^\dagger U(\mathbf z)|0\rangle|^2 = k(\mathbf z, \mathbf z').$$

No experimento principal, usamos a probabilidade exata do simulador. Com shots finitos, $P(0000)$ seria estimada pela frequência de zeros (Seção 5.5). A SVM continua sendo um problema de otimização clássico. O circuito apenas fornece as entradas da matriz de Gram usada por `SVC(kernel="precomputed")`.

**Contagem.** A matriz de Gram de treinamento é simétrica e tem diagonal igual a 1 para estados puros normalizados. Por isso, avaliamos apenas os $n(n-1)/2$ pares $i<j$. A matriz retangular validação × treino serve à seleção, e a matriz teste × treino, à avaliação final. Nenhuma matriz teste × teste é calculada. Os pares são avaliados em lotes com broadcasting de parâmetros do PennyLane, e cada par conta como uma avaliação do kernel.

In [ ]:
def mapa_caracteristicas(z, camadas: int) -> None:
    """U(z): `camadas` repetições de RY(z_i) seguidas de IsingZZ(z_i z_{i+1} / pi) em anel."""
    for _ in range(camadas):

        for i in range(N_QUBITS):
            qml.RY(z[..., i], wires=i)

        for i in range(N_QUBITS):
            j = (i + 1) % N_QUBITS
            qml.IsingZZ(z[..., i] * z[..., j] / np.pi, wires=[i, j])


@qml.qnode(DEV, diff_method=None)
def circuito_inversion_test(z1, z2, camadas: int):
    mapa_caracteristicas(z1, camadas)
    qml.adjoint(mapa_caracteristicas)(z2, camadas)
    return qml.probs(wires=range(N_QUBITS))


@dataclass
class ContadorKernel:
    avaliacoes: int = 0
    shots: int = 0


def kernel_pares(ZA: np.ndarray, ZB: np.ndarray, camadas: int, contador: ContadorKernel) -> np.ndarray:
    """k(ZA[m], ZB[m]) para cada linha m, em lotes."""
    saida = np.empty(len(ZA))
    lote = CONFIG["qsvm_lote"]

    for ini in range(0, len(ZA), lote):
        fim = min(ini + lote, len(ZA))
        probs = np.atleast_2d(circuito_inversion_test(ZA[ini:fim], ZB[ini:fim], camadas))
        saida[ini:fim] = probs[:, 0]

    contador.avaliacoes += len(ZA)

    return saida


def matriz_gram(Z: np.ndarray, camadas: int, contador: ContadorKernel) -> np.ndarray:
    n = len(Z)
    i, j = np.triu_indices(n, k=1)
    K = np.eye(n)
    K[i, j] = kernel_pares(Z[i], Z[j], camadas, contador)
    K[j, i] = K[i, j]

    return K


def matriz_retangular(ZA: np.ndarray, ZB: np.ndarray, camadas: int, contador: ContadorKernel) -> np.ndarray:
    a, b = np.meshgrid(np.arange(len(ZA)), np.arange(len(ZB)), indexing="ij")
    return kernel_pares(ZA[a.ravel()], ZB[b.ravel()], camadas, contador).reshape(len(ZA), len(ZB))


def ajustar_qsvm(p: Particao) -> ResultadoModelo:
    t0 = time.perf_counter()
    contador = ContadorKernel()
    Z_tr, Z_val, Z_te = p.Z["treino"], p.Z["validação"], p.Z["teste"]
    linhas, melhor = [], None
    grams = {}

    for camadas in CONFIG["qsvm_camadas"]:
        K_tr = matriz_gram(Z_tr, camadas, contador)
        K_val = matriz_retangular(Z_val, Z_tr, camadas, contador)
        grams[camadas] = K_tr

        for C in CONFIG["qsvm_C"]:
            svm = SVC(kernel="precomputed", C=C).fit(K_tr, p.y["treino"])
            score = balanced_accuracy_score(p.y["validação"], svm.predict(K_val))
            linhas.append({"camadas do mapa": camadas, "C": C, "acurácia balanceada (validação)": score})

            if melhor is None or score > melhor[0]:
                melhor = (score, camadas, C, svm, K_val)

    _, camadas, C, svm, K_val = melhor
    K_te = matriz_retangular(Z_te, Z_tr, camadas, contador)  # teste x treino
    y_te = svm.predict(K_te)
    specs = qml.specs(circuito_inversion_test)(Z_tr[0], Z_tr[1], camadas)
    recursos = {"qubits": N_QUBITS, "avaliações do kernel": contador.avaliacoes, "shots": "exato (0)", "portas por circuito": specs.resources.num_gates, "profundidade": specs.resources.depth}
    return ResultadoModelo("QSVM (kernel de fidelidade)", {"camadas do mapa": camadas, "C": C}, metricas(p.y["validação"], svm.predict(K_val)), metricas(p.y["teste"], y_te), y_te, int(svm.dual_coef_.size + svm.intercept_.size), time.perf_counter() - t0, recursos, extras={"grade": pd.DataFrame(linhas), "gram": grams[camadas], "svm": svm, "contador": contador})


resultado_qsvm = ajustar_qsvm(P0)
print("Hiperparâmetros selecionados:", resultado_qsvm.hiperparametros)
print("Recursos contados:", resultado_qsvm.recursos)
print(f"Tempo de parede no simulador: {resultado_qsvm.tempo_s:.1f} s")
resultado_qsvm.extras["grade"].round(3)

### 5.3 Propriedades da matriz de Gram no caso ideal

Para o kernel de fidelidade com estados puros e probabilidades exatas, esperamos três propriedades:

- **Simetria:** $k(\mathbf z, \mathbf z') = k(\mathbf z', \mathbf z)$. Verificamos avaliando explicitamente pares nas duas ordens, porque a matriz foi preenchida por simetria.
- **Diagonal:** $k(\mathbf z, \mathbf z) = 1$. Verificamos com o circuito em uma amostra de pontos.
- **Semidefinição positiva:** autovalores $\geq 0$, a menos de erro numérico.

Além disso, $0 \le k \le 1$.

In [ ]:
camadas_sel = resultado_qsvm.hiperparametros["camadas do mapa"]
K_qsvm = resultado_qsvm.extras["gram"]
rng_k = np.random.default_rng(SEMENTE + 1)

amostra = rng_k.choice(len(P0.Z["treino"]), size=min(12, len(P0.Z["treino"])), replace=False)
c_teste = ContadorKernel()

diag = kernel_pares(P0.Z["treino"][amostra], P0.Z["treino"][amostra], camadas_sel, c_teste)
a1, a2 = amostra[:6], amostra[6:12]

ida = kernel_pares(P0.Z["treino"][a1], P0.Z["treino"][a2], camadas_sel, c_teste)
volta = kernel_pares(P0.Z["treino"][a2], P0.Z["treino"][a1], camadas_sel, c_teste)
autovalores = np.linalg.eigvalsh(K_qsvm)

verificar("kernel quântico: diagonal = 1 (circuito)", np.allclose(diag, 1, atol=1e-9))
verificar("kernel quântico: simetria por avaliação nas duas ordens", np.allclose(ida, volta, atol=1e-9))
verificar("kernel quântico: entradas em [0, 1]", K_qsvm.min() >= -1e-12 and K_qsvm.max() <= 1 + 1e-12)
verificar("kernel quântico: Gram de treino semidefinida positiva", autovalores.min() > -1e-9, f"menor autovalor {autovalores.min():.2e}")
print(f"Menor autovalor: {autovalores.min():.3e} | maior: {autovalores.max():.3f} | " f"avaliações extras de verificação: {c_teste.avaliacoes}")

### 5.4 Estrutura da matriz de Gram comparada a um kernel clássico

As matrizes abaixo usam os exemplos de treinamento ordenados por classe. À esquerda, o kernel quântico selecionado. À direita, o kernel RBF com o $\gamma$ selecionado para a SVM RBF. Um kernel útil para classificação tende a mostrar blocos com similaridade maior dentro de cada classe. Valores pequenos fora da diagonal em quase toda a matriz indicariam um kernel concentrado, próximo da identidade, que generaliza mal.

In [ ]:
ordem = np.argsort(P0.y["treino"], kind="stable")
svm_rbf = resultados_classicos["SVM RBF"].extras["modelo"]
gamma_rbf = svm_rbf._gamma
K_rbf = rbf_kernel(P0.Z["treino"], gamma=gamma_rbf)
n0 = int((P0.y["treino"] == 0).sum())
fig, eixos = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.1]})

for eixo, K, titulo in ((eixos[0], K_qsvm, f"kernel quântico ({camadas_sel} camada(s))"), (eixos[1], K_rbf, f"kernel RBF (γ = {gamma_rbf:.3g})")):
    im = eixo.imshow(K[np.ix_(ordem, ordem)], vmin=0, vmax=1, cmap="viridis")
    eixo.axhline(n0 - 0.5, color="white", lw=1)
    eixo.axvline(n0 - 0.5, color="white", lw=1)
    eixo.set(title=titulo, xlabel="exemplo de treino (ordenado por classe)", ylabel="exemplo de treino")
    eixo.grid(False)

fig.colorbar(im, ax=eixos[:2], label="k(z, z')", shrink=0.85)
fora = ~np.eye(len(K_qsvm), dtype=bool)
eixos[2].hist(K_qsvm[fora], bins=40, alpha=0.6, density=True, label="quântico")
eixos[2].hist(K_rbf[fora], bins=40, alpha=0.6, density=True, label="RBF")
eixos[2].set(xlabel="valor do kernel fora da diagonal", ylabel="densidade", title="Distribuição dos valores")
eixos[2].legend()
salvar_figura(fig, "matrizes_kernel")
plt.show()

### 5.5 Kernel com shots finitos

Com $S$ shots, a entrada é estimada por $\hat k = n_{0000}/S$, com $\operatorname{Var}[\hat k] = k(1-k)/S$. Comparamos as estimativas com os valores exatos em um subconjunto de pares do treinamento. Também calculamos uma pequena matriz de Gram estimada para mostrar que a flutuação pode quebrar a semidefinição positiva. Não aplicamos nenhuma correção arbitrária. Uma projeção da matriz de treinamento no cone semidefinido, por exemplo, alteraria a matriz de treino sem alterar de modo coerente as avaliações teste × treino, que também são ruidosas.

In [ ]:
n_pares = min(CONFIG["shots_pares"], len(P0.Z["treino"]) * (len(P0.Z["treino"]) - 1) // 2)
i_s, j_s = np.triu_indices(len(P0.Z["treino"]), k=1)
escolha = rng_k.choice(len(i_s), size=n_pares, replace=False)
Za, Zb = P0.Z["treino"][i_s[escolha]], P0.Z["treino"][j_s[escolha]]
k_exato = K_qsvm[i_s[escolha], j_s[escolha]]
linhas = []

for S in CONFIG["shots_lista"]:
    estimador = qml.set_shots(circuito_inversion_test, shots=S)
    k_est = np.atleast_2d(estimador(Za, Zb, camadas_sel))[:, 0]
    linhas.append({"shots S": S, "erro absoluto médio": np.mean(np.abs(k_est - k_exato)), "desvio padrão teórico médio √(k(1-k)/S)": np.mean(np.sqrt(k_exato * (1 - k_exato) / S)), "pares avaliados": n_pares, "shots totais": n_pares * S})

tabela_shots = pd.DataFrame(linhas)
salvar_tabela(tabela_shots, "kernel_shots")

m = min(30, len(P0.Z["treino"]))
estimador = qml.set_shots(circuito_inversion_test, shots=CONFIG["shots_lista"][0])
ii, jj = np.triu_indices(m, k=1)
K_ruidosa = np.eye(m)

K_ruidosa[ii, jj] = np.atleast_2d(estimador(P0.Z["treino"][ii], P0.Z["treino"][jj], camadas_sel))[:, 0]
K_ruidosa[jj, ii] = K_ruidosa[ii, jj]
min_ruidosa = np.linalg.eigvalsh(K_ruidosa).min()
min_exata = np.linalg.eigvalsh(K_qsvm[:m, :m]).min()

fig, eixo = plt.subplots(figsize=(6.5, 3.8))
eixo.loglog(tabela_shots["shots S"], tabela_shots["erro absoluto médio"], "o-", label="erro absoluto médio (medido)")
eixo.loglog(tabela_shots["shots S"], tabela_shots["desvio padrão teórico médio √(k(1-k)/S)"], "--", label="√(k(1-k)/S) médio (teórico)")
eixo.set(xlabel="shots por avaliação S", ylabel="erro na entrada do kernel", title="Estimativa do kernel com shots finitos")
eixo.legend()
fig.tight_layout()
salvar_figura(fig, "kernel_shots")
plt.show()
print(f"Gram {m}x{m}: menor autovalor exato = {min_exata:.2e}; com {CONFIG['shots_lista'][0]} shots = {min_ruidosa:.2e}")
tabela_shots

<a id="sec-6"></a>
## 6. Classificador variacional híbrido (VQC)

### 6.1 Arquitetura

Um VQC tem quatro componentes:

1. **Codificação:** `AngleEmbedding` com $R_Y(z_i)$ no qubit $i$.
2. **Camadas parametrizadas:** `StronglyEntanglingLayers`. Cada camada aplica uma rotação geral $\mathrm{Rot}(\alpha,\beta,\gamma) = R_Z(\gamma)R_Y(\beta)R_Z(\alpha)$ em cada qubit, seguida de CNOTs em anel, que fazem o emaranhamento. Uma camada tem $4 \times 3 = 12$ parâmetros treináveis.
3. **Saída:** o valor esperado $\langle Z_0\rangle \in [-1, 1]$ do qubit 0, convertido em probabilidade $p(y=1\mid\mathbf z) = (1 - \langle Z_0\rangle)/2$, que é a probabilidade de medir $|1\rangle$ no qubit 0. A previsão é a classe 1 quando $p > 1/2$.
4. **Perda e otimização clássica:** entropia cruzada binária com $p$ recortado em $[\varepsilon, 1-\varepsilon]$, $\varepsilon = 10^{-7}$, para evitar $\log 0$. O otimizador é Adam (`qml.AdamOptimizer`) com lote completo.

**Relação com o MLP.** Os dois modelos são funções parametrizadas treinadas por gradiente, com a mesma perda. No MLP, o gradiente vem da retropropagação sobre operações clássicas. No VQC, cada derivada de um circuito em hardware exige execuções adicionais do circuito (Seção 6.2). Neste notebook, o treinamento principal usa diferenciação automática do simulador (`diff_method="backprop"`), que percorre o vetor de estado simulado. Esse recurso não está disponível em hardware quântico e não equivale à retropropagação convencional executada em um processador quântico.

**Codificação única e recarregamento.** Comparamos duas variantes com o mesmo número de camadas $L$ e, portanto, o mesmo número de parâmetros treináveis, $12L$:

- codificação única: $W_L\cdots W_1\,S(\mathbf z)$;
- recarregamento de dados (data re-uploading): $W_L S(\mathbf z)\cdots W_1 S(\mathbf z)$.

Com $L = 1$, as duas variantes são o mesmo circuito, e apenas uma é treinada. Os hiperparâmetros são $L \in$ `CONFIG["vqc_camadas"]` e a variante. A época de parada é a de menor perda de validação. Tudo é escolhido pela validação.

In [ ]:
EPS_PERDA = 1e-7

def operacoes_vqc(z, pesos, recarregar: bool) -> None:

    for camada in range(pesos.shape[0]):
        if recarregar or camada == 0:
            qml.AngleEmbedding(z, wires=range(N_QUBITS), rotation="Y")
        qml.StronglyEntanglingLayers(pesos[camada:camada + 1], wires=range(N_QUBITS))

@qml.qnode(DEV, diff_method="backprop", interface="autograd")
def vqc_expval(z, pesos, recarregar: bool):
    operacoes_vqc(z, pesos, recarregar)
    return qml.expval(qml.PauliZ(0))


@qml.qnode(DEV, diff_method="parameter-shift", interface="autograd")
def vqc_expval_ps(z, pesos, recarregar: bool):
    operacoes_vqc(z, pesos, recarregar)
    return qml.expval(qml.PauliZ(0))


def probabilidade_classe1(Z, pesos, recarregar: bool):
    return (1 - vqc_expval(Z, pesos, recarregar)) / 2


def perda_bce(pesos, Z, y, recarregar: bool):
    p = pnp.clip(probabilidade_classe1(Z, pesos, recarregar), EPS_PERDA, 1 - EPS_PERDA)
    return -pnp.mean(y * pnp.log(p) + (1 - y) * pnp.log(1 - p))


def prever_vqc(Z, pesos, recarregar: bool) -> np.ndarray:
    return (np.asarray(probabilidade_classe1(Z, pesos, recarregar)) > 0.5).astype(int)


def treinar_vqc(p: Particao, camadas: int, recarregar: bool) -> dict:
    rng = np.random.default_rng(p.semente + 100 * camadas + int(recarregar))
    pesos = pnp.array(rng.uniform(0, 2 * np.pi, (camadas, N_QUBITS, 3)), requires_grad=True)
    Z_tr = pnp.array(p.Z["treino"], requires_grad=False)
    Z_val = pnp.array(p.Z["validação"], requires_grad=False)
    y_tr = pnp.array(p.y["treino"], requires_grad=False)
    y_val = pnp.array(p.y["validação"], requires_grad=False)
    otimizador = qml.AdamOptimizer(CONFIG["vqc_taxa"])
    historico = {"perda treino": [], "perda validação": [], "acurácia balanceada validação": []}
    melhor = (np.inf, None, 0)

    for epoca in range(1, CONFIG["vqc_epocas"] + 1):
        pesos = otimizador.step(lambda w: perda_bce(w, Z_tr, y_tr, recarregar), pesos)
        perda_val = float(perda_bce(pesos, Z_val, y_val, recarregar))
        historico["perda treino"].append(float(perda_bce(pesos, Z_tr, y_tr, recarregar)))
        historico["perda validação"].append(perda_val)
        historico["acurácia balanceada validação"].append(
            balanced_accuracy_score(p.y["validação"], prever_vqc(Z_val, pesos, recarregar)))
        if perda_val < melhor[0]:
            melhor = (perda_val, pnp.array(pesos, requires_grad=False), epoca)

    return {"pesos": melhor[1], "época": melhor[2], "historico": historico}


def ajustar_vqc(p: Particao) -> ResultadoModelo:
    t0 = time.perf_counter()
    linhas, treinos, escolhido = [], {}, None

    for camadas, recarregar in itertools.product(CONFIG["vqc_camadas"], (False, True)):

        if camadas == 1 and recarregar:
            continue  # com L = 1 as duas variantes são o mesmo circuito

        tr = treinar_vqc(p, camadas, recarregar)
        score = balanced_accuracy_score(p.y["validação"], prever_vqc(p.Z["validação"], tr["pesos"], recarregar))
        nome = f"L={camadas}, " + ("recarregamento" if recarregar else "codificação única")
        treinos[nome] = tr

        linhas.append({"configuração": nome, "parâmetros treináveis": 12 * camadas, "época selecionada": tr["época"], "perda validação": tr["historico"]["perda validação"][tr["época"] - 1], "acurácia balanceada (validação)": score})

        if escolhido is None or score > escolhido[0]:
            escolhido = (score, camadas, recarregar, nome)

    _, camadas, recarregar, nome = escolhido
    pesos = treinos[nome]["pesos"]
    y_te = prever_vqc(p.Z["teste"], pesos, recarregar)
    n_tr, epocas = len(p.y["treino"]), CONFIG["vqc_epocas"]
    n_conf = len(linhas)

    recursos = {"qubits": N_QUBITS,
                "avaliações diretas do circuito (treino+validação, todas as configurações)": n_conf * epocas * (2 * n_tr + len(p.y["validação"])),
                "shots": "exato (0)",
                "estimativa por parameter-shift em hardware (por época, modelo selecionado)": 2 * 12 * camadas * n_tr}

    return ResultadoModelo("VQC", {"camadas": camadas, "variante": nome, "época": treinos[nome]["época"]}, metricas(p.y["validação"], prever_vqc(p.Z["validação"], pesos, recarregar)), metricas(p.y["teste"], y_te), y_te, 12 * camadas, time.perf_counter() - t0, recursos, extras={"grade": pd.DataFrame(linhas), "treinos": treinos, "pesos": pesos, "recarregar": recarregar})


resultado_vqc = ajustar_vqc(P0)
print("Configuração selecionada:", resultado_vqc.hiperparametros)
print(f"Tempo de parede no simulador: {resultado_vqc.tempo_s:.1f} s")
resultado_vqc.extras["grade"].round(4)

In [ ]:
fig = qml.draw_mpl(vqc_expval, level="device", decimals=None, style="black_white")(P0.Z["treino"][0], resultado_vqc.extras["pesos"], resultado_vqc.extras["recarregar"])[0]
fig.suptitle(f"Circuito do VQC selecionado ({resultado_vqc.hiperparametros['variante']}), portas expandidas")
salvar_figura(fig, "vqc_circuito")
plt.show()

fig, eixos = plt.subplots(1, 2, figsize=(13, 4))

for nome, tr in resultado_vqc.extras["treinos"].items():
    epocas = np.arange(1, len(tr["historico"]["perda treino"]) + 1)
    linha, = eixos[0].plot(epocas, tr["historico"]["perda treino"], label=f"{nome}: treino")
    eixos[0].plot(epocas, tr["historico"]["perda validação"], "--", color=linha.get_color(), label=f"{nome}: validação")
    eixos[1].plot(epocas, tr["historico"]["acurácia balanceada validação"], color=linha.get_color(), label=nome)

eixos[0].set(xlabel="época", ylabel="entropia cruzada binária", title="VQC: perda de treino (contínua) e validação (tracejada)")
eixos[1].set(xlabel="época", ylabel="acurácia balanceada", title="VQC: acurácia balanceada na validação", ylim=(0, 1.02))
eixos[0].legend(fontsize=7)
eixos[1].legend(fontsize=7)
fig.tight_layout()
salvar_figura(fig, "vqc_curvas")
plt.show()

### 6.2 Regra de deslocamento de parâmetros

Para uma porta $e^{-i\theta G/2}$ com gerador $G$ de autovalores $\pm1$, como $R_X$, $R_Y$ e $R_Z$, o valor esperado $f(\theta) = \langle O\rangle_\theta$ é uma senoide em $\theta$. Sua derivada é exata:

$$\frac{\partial f}{\partial\theta} = \frac{f(\theta + \pi/2) - f(\theta - \pi/2)}{2}.$$

Em hardware, cada derivada parcial custa duas execuções adicionais do circuito, cada uma estimada com shots. Exemplo: com $R_X(\theta)|0\rangle$ e $O = Z$, temos $f = \cos\theta$ e $f' = -\sin\theta$. No VQC, as portas $\mathrm{Rot}$ são decompostas em $R_Z R_Y R_Z$, e a regra é aplicada a cada ângulo. Abaixo, comparamos o gradiente por parameter-shift com a diferenciação do simulador e com diferenças finitas centrais.

In [ ]:
@qml.qnode(DEV, diff_method="parameter-shift", interface="autograd")
def exemplo_rx(theta):
    qml.RX(theta, wires=0)
    return qml.expval(qml.PauliZ(0))


theta = pnp.array(0.7, requires_grad=True)
deslocamento = (exemplo_rx(theta + np.pi / 2) - exemplo_rx(theta - np.pi / 2)) / 2
verificar("parameter-shift manual em RX = -sin(theta)", abs(float(deslocamento) + math.sin(0.7)) < 1e-12)
verificar("qml.grad (parameter-shift) em RX = -sin(theta)", abs(float(qml.grad(exemplo_rx)(theta)) + math.sin(0.7)) < 1e-12)

pesos_teste = pnp.array(np.random.default_rng(SEMENTE).uniform(0, 2 * np.pi, (2, N_QUBITS, 3)), requires_grad=True)
z_teste = pnp.array(P0.Z["treino"][0], requires_grad=False)
g_ps = qml.grad(vqc_expval_ps, argnums=1)(z_teste, pesos_teste, True)
g_bp = qml.grad(vqc_expval, argnums=1)(z_teste, pesos_teste, True)
h = 1e-5
g_df = np.zeros(pesos_teste.shape)

for idx in np.ndindex(pesos_teste.shape):
    mais, menos = np.array(pesos_teste), np.array(pesos_teste)
    mais[idx] += h
    menos[idx] -= h
    g_df[idx] = (float(vqc_expval(z_teste, mais, True)) - float(vqc_expval(z_teste, menos, True))) / (2 * h)

verificar("VQC: gradiente parameter-shift = backprop (tol 1e-8)", np.allclose(g_ps, g_bp, atol=1e-8))
verificar("VQC: gradiente parameter-shift = diferenças finitas centrais (tol 1e-6)", np.allclose(g_ps, g_df, atol=1e-6))
print(f"Máx |PS - backprop| = {np.max(np.abs(g_ps - g_bp)):.2e}; máx |PS - DF| = {np.max(np.abs(g_ps - g_df)):.2e}")

### 6.3 Recarregamento e frequências acessíveis

Quando a entrada $z$ entra apenas por rotações $R_Y(z) = e^{-izY/2}$, cujo gerador $Y/2$ tem autovalores $\pm\tfrac12$, a saída do modelo é uma série de Fourier finita em $z$. As frequências acessíveis são as diferenças entre somas de autovalores do gerador. Com uma única codificação, a frequência máxima em cada atributo é 1. Com $L$ recarregamentos, chega a $L$.

A figura mostra, para $L = 2$, o maior módulo de cada coeficiente de Fourier de $f(z_k) = \langle Z_0\rangle$ entre 10 sorteios independentes de pesos. Os outros atributos ficam fixos em $\pi/2$. Mostramos dois casos: o atributo do qubit medido ($z_0$) e o de outro qubit ($z_1$). O conjunto de frequências acessíveis é apenas um limite superior. Nesta arquitetura, o coeficiente de frequência 2 em $z_0$ se anula em todos os sorteios, enquanto em $z_1$ ele aparece. Isso ilustra que o recarregamento amplia o espectro possível, mas não torna todos os coeficientes livres e independentes. Ele também não garante melhor classificação nem implica vantagem computacional.

In [ ]:
grade_z = np.linspace(0, 2 * np.pi, 64, endpoint=False)
rng_freq = np.random.default_rng(SEMENTE + 2)
sorteios_pesos = [rng_freq.uniform(0, 2 * np.pi, (2, N_QUBITS, 3)) for _ in range(10)]
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
espectros = {}

for eixo, atributo in zip(eixos, (0, 1)):
    Z_grade = np.tile(np.pi / 2, (len(grade_z), N_QUBITS))
    Z_grade[:, atributo] = grade_z

    for recarregar, rotulo in ((False, "codificação única"), (True, "recarregamento")):
        coef = np.max([np.abs(np.fft.rfft(np.asarray(vqc_expval(Z_grade, w, recarregar)))) / len(grade_z) for w in sorteios_pesos], axis=0)
        espectros[(atributo, rotulo)] = coef
        eixo.bar(np.arange(6) + (0.2 if recarregar else -0.2), coef[:6], width=0.4, label=rotulo)

    eixo.set(xlabel="frequência inteira ω", title=f"Espectro em z_{atributo} (L = 2, máximo em 10 sorteios de pesos)")
    eixo.set_yscale("symlog", linthresh=1e-6)
    eixo.legend()

eixos[0].set_ylabel("max |c_ω| (escala symlog)")
fig.tight_layout()
salvar_figura(fig, "vqc_frequencias")
plt.show()

for atributo in (0, 1):
    verificar(f"codificação única: sem frequências > 1 em z_{atributo}", espectros[(atributo, "codificação única")][2:].max() < 1e-10)
    verificar(f"recarregamento L=2: sem frequências > 2 em z_{atributo}", espectros[(atributo, "recarregamento")][3:].max() < 1e-10)

print("max |c_2| com recarregamento: z_0 = %.2e, z_1 = %.2e" % (espectros[(0, "recarregamento")][2], espectros[(1, "recarregamento")][2]))

<a id="sec-7"></a>
## 7. Experimento de treinabilidade: platôs estéreis

Esta seção é um experimento controlado de treinabilidade, complementar à classificação e independente do dataset. Um platô estéril é a concentração exponencial dos gradientes em torno de zero quando os parâmetros são sorteados de uma distribuição: $\operatorname{Var}[\partial_k C]$ decresce exponencialmente com o número de qubits. Não se conclui que há platô apenas porque um treinamento falhou, nem porque a média dos gradientes é próxima de zero. A média é zero por simetria em muitos casos, com ou sem platô. O que importa é a variância em função do tamanho.

### 7.1 Exemplo analítico: custo global contra custo local

Seguimos o exemplo de Cerezo et al. (2021): $V(\boldsymbol\theta) = \bigotimes_{j=1}^n R_X(\theta_j)$ aplicado a $|0\rangle^{\otimes n}$, com $\theta_j$ independentes e uniformes em $[0, 2\pi)$. As grandezas são:

- **Custo global**, baseado na probabilidade conjunta de todos os zeros: $C_G = 1 - P(0\cdots0) = 1 - \prod_j \cos^2(\theta_j/2)$.
- **Custo local**, a média das probabilidades individuais de obter 1: $C_L = \tfrac1n\sum_j P_j(1) = 1 - \tfrac1n\sum_j\cos^2(\theta_j/2)$.
- **Componente do gradiente:** sempre $\partial/\partial\theta_1$, calculada pela regra de deslocamento de parâmetros com circuitos PennyLane.

Ambos os custos estão normalizados em $[0,1]$. Neste exemplo, o cálculo direto dá

$$\operatorname{Var}\!\left[\frac{\partial C_G}{\partial\theta_1}\right] = \frac18\left(\frac38\right)^{n-1}, \qquad \operatorname{Var}\!\left[\frac{\partial C_L}{\partial\theta_1}\right] = \frac{1}{8n^2},$$

usando $\mathbb E[\sin^2\theta/4] = 1/8$ e $\mathbb E[\cos^4(\theta/2)] = 3/8$. As expressões coincidem com as reportadas por Cerezo et al. Variância decrescente significa concentração: um gradiente amostrado fica cada vez mais próximo de zero, e estimá-lo com shots exige cada vez mais repetições.

**Estimação da variância.** Os gradientes de cada circuito são conferidos amostra a amostra contra as derivadas analíticas. A variância é estimada pela variância amostral sobre `CONFIG["bp_amostras"]` inicializações. Para o custo global, a distribuição do gradiente tem caudas pesadas: a maior parte das amostras é quase nula, e poucas amostras grandes dominam a variância. A razão $\mathbb E[g^4]/\mathbb E[g^2]^2$ cresce como $(35/24)^{n-1}$, e a variância amostral passa a exigir muito mais inicializações para ter a mesma precisão relativa. Por isso, para $n$ grande, a estimativa global pode se afastar da curva analítica. Essa dificuldade é outra face da concentração.

In [ ]:
@qml.qnode(DEV, diff_method=None)
def custos_produto_rx(theta, n: int):
    for j in range(n):
        qml.RX(theta[..., j], wires=j)

    return qml.probs(wires=range(n)), *[qml.expval(qml.PauliZ(j)) for j in range(n)]


def custos_global_local(saidas) -> tuple[np.ndarray, np.ndarray]:
    probs, *expvals = saidas
    c_global = 1 - np.atleast_2d(probs)[:, 0]
    c_local = np.mean([(1 - np.asarray(e)) / 2 for e in expvals], axis=0)

    return c_global, c_local


def gradiente_primeiro_parametro(circuito: Callable, theta: np.ndarray, *args) -> tuple[np.ndarray, np.ndarray]:
    """d/d(theta_1) pelos custos em theta_1 +- pi/2 (regra de deslocamento de parâmetros)."""
    mais, menos = theta.copy(), theta.copy()
    mais[:, 0] += np.pi / 2
    menos[:, 0] -= np.pi / 2
    gm, lm = custos_global_local(circuito(mais, *args))
    gn, ln = custos_global_local(circuito(menos, *args))

    return (gm - gn) / 2, (lm - ln) / 2


rng_bp = np.random.default_rng(SEMENTE + 7)
linhas, gradientes = [], {}

for n in CONFIG["bp_qubits"]:
    theta = rng_bp.uniform(0, 2 * np.pi, (CONFIG["bp_amostras"], n))
    g_glob, g_loc = gradiente_primeiro_parametro(custos_produto_rx, theta, n)
    gradientes[n] = (g_glob, g_loc)
    # Referência independente, amostra a amostra: derivadas analíticas de C_G e C_L.
    g_glob_ref = 0.5 * np.sin(theta[:, 0]) * np.prod(np.cos(theta[:, 1:] / 2) ** 2, axis=1)
    g_loc_ref = np.sin(theta[:, 0]) / (2 * n)
    verificar(f"platô n={n}: gradientes dos circuitos = derivadas analíticas (tol 1e-10)", np.allclose(g_glob, g_glob_ref, atol=1e-10) and np.allclose(g_loc, g_loc_ref, atol=1e-10))
    linhas.append({"n qubits": n, "amostras": len(theta),
                   "Var[∂C_G] estimada": g_glob.var(ddof=1), "Var[∂C_G] analítica": (1 / 8) * (3 / 8) ** (n - 1),
                   "Var[∂C_L] estimada": g_loc.var(ddof=1), "Var[∂C_L] analítica": 1 / (8 * n**2),
                   "média ∂C_G": g_glob.mean(), "média ∂C_L": g_loc.mean()})

tabela_bp = pd.DataFrame(linhas)
salvar_tabela(tabela_bp, "plato_produto_rx")
razoes_locais = tabela_bp["Var[∂C_L] estimada"] / tabela_bp["Var[∂C_L] analítica"]
verificar("platô: variância do custo local dentro de um fator 1,5 da analítica", np.all(np.abs(np.log(razoes_locais)) < np.log(1.5)), f"razões: {np.round(razoes_locais.to_numpy(), 2)}")
tabela_bp["razão estimada/analítica (global)"] = tabela_bp["Var[∂C_G] estimada"] / tabela_bp["Var[∂C_G] analítica"]

theta_v = pnp.array(np.random.default_rng(1).uniform(0, 2 * np.pi, 3), requires_grad=True)

@qml.qnode(DEV, diff_method="backprop", interface="autograd")
def custo_global_autodiff(theta):
    for j in range(3):
        qml.RX(theta[j], wires=j)
    return qml.probs(wires=range(3))

g_auto = qml.grad(lambda t: 1 - custo_global_autodiff(t)[0])(theta_v)[0]
g_shift, _ = gradiente_primeiro_parametro(custos_produto_rx, np.array([theta_v]), 3)
verificar("platô: deslocamento de parâmetros = diferenciação automática (n=3)", abs(g_auto - g_shift[0]) < 1e-10)

fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
ns = tabela_bp["n qubits"]
eixos[0].semilogy(ns, tabela_bp["Var[∂C_G] estimada"], "o", color="C3", label="global (estimado)")
eixos[0].semilogy(ns, tabela_bp["Var[∂C_G] analítica"], "-", color="C3", lw=1, label="global: (1/8)(3/8)^(n-1) (analítico)")
eixos[0].semilogy(ns, tabela_bp["Var[∂C_L] estimada"], "s", color="C0", label="local (estimado)")
eixos[0].semilogy(ns, tabela_bp["Var[∂C_L] analítica"], "-", color="C0", lw=1, label="local: 1/(8n²) (analítico)")
eixos[0].set(xlabel="número de qubits n", ylabel="Var[∂C/∂θ₁] (escala log)", title="Variância do gradiente")
eixos[0].legend(fontsize=7)
bins_log = np.linspace(-12, 0, 49)

for eixo, (indice, nome) in zip(eixos[1:], ((0, "global C_G"), (1, "local C_L"))):

    for n in CONFIG["bp_qubits"]:
        valores = np.log10(np.abs(gradientes[n][indice]) + 1e-16)
        eixo.hist(valores, bins=bins_log, histtype="step", lw=1.5, label=f"n = {n}")

    eixo.set(xlabel="log10 |∂C/∂θ₁|", ylabel="número de inicializações", title=f"Magnitude do gradiente, custo {nome}")
    eixo.legend(fontsize=8)

fig.tight_layout()
salvar_figura(fig, "plato_global_local")
plt.show()
tabela_bp

### 7.2 Extensão: ansatz com emaranhamento e profundidade variável

O ansatz tem $D$ camadas. Cada camada aplica $R_Y(\theta_{d,j})$ a todos os qubits, seguido de CZ em cadeia, entre $j$ e $j+1$. Os parâmetros são sorteados uniformemente em $[0, 2\pi)$. Os custos e a componente do gradiente ($\partial/\partial\theta_{1,1}$, a primeira rotação) são os mesmos da Seção 7.1.

Este estudo depende da arquitetura, da inicialização e do custo escolhidos. Ele não mostra que aumentar qubits ou camadas produz o mesmo efeito em qualquer modelo. Os resultados dizem respeito apenas a este ansatz.

In [ ]:
@qml.qnode(DEV, diff_method=None)
def custos_ansatz_emaranhado(theta, n: int, profundidade: int):
    theta = theta.reshape(theta.shape[0], profundidade, n) if theta.ndim == 2 else theta.reshape(profundidade, n)

    for d in range(profundidade):

        for j in range(n):
            qml.RY(theta[..., d, j], wires=j)

        for j in range(n - 1):
            qml.CZ(wires=[j, j + 1])

    return qml.probs(wires=range(n)), *[qml.expval(qml.PauliZ(j)) for j in range(n)]


linhas = []

for n in CONFIG["bp_ext_qubits"]:

    for prof in CONFIG["bp_ext_profundidades"]:
        theta = rng_bp.uniform(0, 2 * np.pi, (CONFIG["bp_ext_amostras"], prof * n))
        g_glob, g_loc = gradiente_primeiro_parametro(custos_ansatz_emaranhado, theta, n, prof)
        linhas.append({"n qubits": n, "profundidade D": prof, "Var[∂C_G]": g_glob.var(ddof=1), "Var[∂C_L]": g_loc.var(ddof=1)})

tabela_bp_ext = pd.DataFrame(linhas)
salvar_tabela(tabela_bp_ext, "plato_ansatz_emaranhado")

fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for eixo, col, titulo in ((eixos[0], "Var[∂C_G]", "custo global C_G"), (eixos[1], "Var[∂C_L]", "custo local C_L")):

    for prof, grupo in tabela_bp_ext.groupby("profundidade D"):
        eixo.semilogy(grupo["n qubits"], grupo[col], "o-", label=f"D = {prof}")

    eixo.set(xlabel="número de qubits n", title=f"Ansatz RY + CZ em cadeia: {titulo}")
    eixo.legend(title="camadas")

eixos[0].set_ylabel("Var[∂C/∂θ₁,₁] (escala log)")
fig.suptitle(f"Estudo dependente da arquitetura e da inicialização (uniforme em [0, 2π), {CONFIG['bp_ext_amostras']} amostras)")
fig.tight_layout()
salvar_figura(fig, "plato_ansatz_emaranhado")
plt.show()
tabela_bp_ext.pivot(index="n qubits", columns="profundidade D", values=["Var[∂C_G]", "Var[∂C_L]"])

<a id="sec-8"></a>
## 8. Comparação final

### 8.1 Avaliação no conjunto de teste

Todos os modelos foram selecionados apenas com a validação. A avaliação a seguir usa o conjunto de teste da partição principal. As matrizes de confusão têm as linhas como classe verdadeira e as colunas como classe prevista.

In [ ]:
resultados_principais = {**resultados_classicos, "QSVM": resultado_qsvm, "VQC": resultado_vqc}
resultados_controle = ajustar_classicos(P0, chave="C", sufixo=" [controle]") if P0.C is not None else {}
todos_resultados = {**resultados_principais, **resultados_controle}

n_modelos = len(todos_resultados)
colunas = 4
linhas_fig = math.ceil(n_modelos / colunas)
fig, eixos = plt.subplots(linhas_fig, colunas, figsize=(3.6 * colunas, 3.3 * linhas_fig), squeeze=False)

for eixo, (nome, r) in zip(eixos.ravel(), todos_resultados.items()):
    cm = confusion_matrix(P0.y["teste"], r.y_pred_teste, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=[ROTULOS_CLASSES[0], ROTULOS_CLASSES[1]]).plot(
        ax=eixo, colorbar=False, cmap="Blues")
    eixo.set_title(nome, fontsize=9)
    eixo.grid(False)
    eixo.set_xlabel("previsto", fontsize=8)
    eixo.set_ylabel("verdadeiro", fontsize=8)

for eixo in eixos.ravel()[n_modelos:]:
    eixo.axis("off")

fig.suptitle(f"Matrizes de confusão no teste (n = {len(P0.y['teste'])}, semente {P0.semente})")
fig.tight_layout()
salvar_figura(fig, "matrizes_confusao")
plt.show()

### 8.2 Tabela final

As colunas de recursos quânticos registram o que foi efetivamente contado no simulador: avaliações do kernel e avaliações diretas do circuito, todas com probabilidades exatas (shots = 0). A coluna de estimativa para hardware indica quantas execuções o parameter-shift exigiria por época e não foi executada. O tempo é o tempo de parede da seleção e do ajuste, incluindo todas as configurações da grade, neste computador. Ele mede o custo da simulação clássica, não o de um processador quântico.

O erro padrão binomial $\sqrt{a(1-a)/n_{\text{teste}}}$ indica a incerteza da acurácia $a$ devida ao tamanho do teste. Ele é uma aproximação e não é um intervalo de confiança calculado.

In [ ]:
def tabela_resultados(resultados: dict[str, ResultadoModelo], p: Particao) -> pd.DataFrame:
    linhas = []

    for nome, r in resultados.items():
        acc = r.metricas_teste["acurácia"]
        linhas.append({
            "modelo": nome,
            "representação": "controle (atributos originais padronizados)" if "[controle]" in nome else "principal ([0, π]^4)",
            "acurácia (teste)": acc,
            "erro padrão binomial": math.sqrt(acc * (1 - acc) / len(p.y["teste"])),
            "acurácia balanceada (teste)": r.metricas_teste["acurácia balanceada"],
            "F1 (teste)": r.metricas_teste["F1"],
            "acurácia balanceada (validação)": r.metricas_validacao["acurácia balanceada"],
            "n treino / val / teste": f"{len(p.y['treino'])} / {len(p.y['validação'])} / {len(p.y['teste'])}",
            "parâmetros": r.parametros,
            "tempo de seleção e ajuste (s)": r.tempo_s,
            "qubits": r.recursos.get("qubits", 0),
            "avaliações quânticas contadas": r.recursos.get("avaliações do kernel",
                                                           r.recursos.get("avaliações diretas do circuito (treino+validação, todas as configurações)", 0)),
            "shots": r.recursos.get("shots", "—"),
            "parameter-shift em hardware por época (estimativa)": r.recursos.get(
                "estimativa por parameter-shift em hardware (por época, modelo selecionado)", "—"),
            "hiperparâmetros": str(r.hiperparametros),
        })

    return pd.DataFrame(linhas)


tabela_final = tabela_resultados(todos_resultados, P0)
salvar_tabela(tabela_final, "comparacao_final")

with pd.option_context("display.max_columns", 30, "display.width", 250):
    display(tabela_final.drop(columns=["hiperparâmetros"]).round(3))

In [ ]:
fig, eixo = plt.subplots(figsize=(11, 4))
nomes = tabela_final["modelo"]
pos = np.arange(len(nomes))

for k, (col, cor) in enumerate((("acurácia (teste)", "C0"), ("acurácia balanceada (teste)", "C1"), ("F1 (teste)", "C2"))):
    erro = tabela_final["erro padrão binomial"] if col == "acurácia (teste)" else None
    eixo.bar(pos + (k - 1) * 0.27, tabela_final[col], width=0.27, color=cor, label=col, yerr=erro, capsize=3)

eixo.set(xticks=pos, ylim=(0, 1.05), ylabel="valor da métrica", title=f"Métricas no teste (semente {P0.semente}; barras de erro: erro padrão binomial da acurácia)")
eixo.set_xticklabels(nomes, rotation=20, ha="right", fontsize=8)
eixo.axhline(0.5, color="gray", lw=0.8, ls=":")
eixo.legend(fontsize=8, loc="lower right")
fig.tight_layout()
salvar_figura(fig, "metricas_teste")
plt.show()

### 8.3 Várias sementes (modo completo)

No modo completo, o protocolo inteiro é repetido para cada semente de divisão: transformações, seleção e teste. A tabela mostra a média e o desvio padrão entre execuções. Esse desvio descreve a dispersão causada pela divisão dos dados e pela inicialização. Ele não é um intervalo de confiança. No modo rápido, apenas a semente principal é executada e esta célula informa isso.

In [ ]:
def executar_protocolo(p: Particao) -> dict[str, ResultadoModelo]:
    r = {**ajustar_classicos(p), "QSVM": ajustar_qsvm(p), "VQC": ajustar_vqc(p)}

    if p.C is not None:
        r.update(ajustar_classicos(p, chave="C", sufixo=" [controle]"))

    return r

if len(particoes) > 1:
    por_semente = [tabela_resultados(todos_resultados, P0).assign(semente=P0.semente)]

    for s, p in particoes.items():
        if s != P0.semente:
            por_semente.append(tabela_resultados(executar_protocolo(p), p).assign(semente=s))

    tabela_sementes = pd.concat(por_semente)
    salvar_tabela(tabela_sementes, "resultados_por_semente")
    resumo_sementes = tabela_sementes.groupby("modelo", sort=False)[["acurácia (teste)", "acurácia balanceada (teste)", "F1 (teste)"]].agg(["mean", "std"])
    display(resumo_sementes.round(3))

else:
    print("Modo rápido: apenas a semente principal foi executada. Use TCC_MODO=completo para várias sementes.")

<a id="sec-9"></a>
## 9. Verificações, ambiente e tempo de execução

As verificações abaixo foram executadas ao longo do notebook. Cada uma interromperia a execução em caso de falha. As tolerâncias aparecem no nome ou na célula correspondente. A tolerância padrão é $10^{-9}$ para propriedades do kernel e $10^{-10}$ a $10^{-12}$ para identidades exatas.

In [ ]:
tabela_verificacoes = pd.DataFrame(REGISTRO_VERIFICACOES)
salvar_tabela(tabela_verificacoes, "verificacoes")
print(f"Verificações aprovadas: {int(tabela_verificacoes['aprovada'].sum())} de {len(tabela_verificacoes)}")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 110):
    display(tabela_verificacoes)

In [ ]:
tempo_total = time.perf_counter() - T_INICIO_NOTEBOOK
ambiente = pd.DataFrame([
    {"item": "modo", "valor": MODO},
    {"item": "tempo total do notebook (s)", "valor": f"{tempo_total:.1f}"},
    {"item": "Python", "valor": platform.python_version()},
    {"item": "sistema", "valor": f"{platform.system()} {platform.release()} ({platform.machine()})"},
    {"item": "processadores lógicos", "valor": str(os.cpu_count())},
    {"item": "PennyLane", "valor": qml.__version__},
    {"item": "scikit-learn", "valor": sklearn.__version__},
    {"item": "NumPy", "valor": np.__version__},
    {"item": "SciPy", "valor": scipy.__version__},
    {"item": "pandas", "valor": pd.__version__},
    {"item": "Matplotlib", "valor": matplotlib.__version__},
])
salvar_tabela(ambiente, "ambiente_execucao")
ambiente